# Full run: results on the real models

Every SWE-bench Lite item and 300 LoCoMo questions, through each condition, on **Llama 3.1 8B** (vLLM on a Modal H100) and **Muse Spark 1.2** (Meta Model API). Then the team replay.

**The north star:** for a team sharing one codebase, the gateway matches the accuracy of sending context straight to Muse while sending **≤ 10%** of Muse's tokens. The pass rule is **within 3 points of `muse_direct`, at ≤ 10% of its Muse tokens**.

| Condition | What Muse sees |
|---|---|
| `muse_direct` | the whole context, no gateway (SWE-bench: ~11K tokens of retrieved code; LoCoMo: the full conversation, ~18K tokens) |
| `full` | the cascade gate at 10% (after `scripts/gate_run.sh`): Llama drafts, the gate keeps the draft or escalates |
| `escalate_raw` | always escalates, with the raw pages behind the top-5 memories (after `scripts/gate_run.sh`) |
| `escalate_compact` | always escalates, with the top-5 memory summaries |
| `rule_gate` | the hand-written gate: escalates long or code-heavy questions |
| `llama_only` | never: Llama answers with memory |

Sections:
1. What was run
2. The north star, per benchmark
3. Cost and latency
4. LoCoMo: where the gap comes from
5. SWE-bench: what its accuracy measures
6. Routing headroom: the best any gate could do
7. Team scaling (replay)
8. The learned gate
9. Summary

Everything is read from `eval/results/`. Nothing here calls a model or needs the database. `full` and `escalate_raw` appear in every table once `scripts/gate_run.sh` has run; just rerun the notebook.

In [ ]:
RUN_ID = "full1"   # the FULL_ID scripts/full_run.sh used
N, SEED = 300, 0   # its FULL_N, and eval.run's default seed

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "eval").is_dir() else Path.cwd().parent
os.chdir(ROOT)  # so .env and the repo's packages are found
sys.path.insert(0, str(ROOT))

from eval import locomo  # noqa: E402
from gateway.config import Settings  # noqa: E402

RESULTS = ROOT / "eval" / "results"
BENCHES = ["swebench", "locomo"]
ORDER = ["muse_direct", "full", "escalate_raw", "escalate_compact", "rule_gate", "llama_only"]
HEADLINE = {"swebench": "hit", "locomo": "correct"}
CATEGORIES = {1: "multi-hop", 2: "temporal", 3: "open-domain", 4: "single-hop"}
MARGIN, BUDGET = 3.0, 0.10  # points of accuracy, share of muse_direct's Muse tokens
settings = Settings()
PRICE_IN = settings.muse_input_price_per_m / 1e6
PRICE_OUT = settings.muse_output_price_per_m / 1e6

# Validated colours (dataviz reference palette; all three pass all-pairs CVD checks).
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 12,
    "axes.titleweight": "bold", "font.family": ["Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 110,
})
pd.set_option("display.max_colwidth", 80)


def run_id(bench: str, condition: str) -> str:
    return f"{RUN_ID}-{bench}-{condition}"


runs = {}  # (bench, condition) -> summary
for bench in BENCHES:
    for c in ORDER:
        path = RESULTS / f"{bench}_{c}_{run_id(bench, c)}.json"
        if path.exists():
            runs[bench, c] = json.loads(path.read_text())
conditions = {b: [c for c in ORDER if (b, c) in runs] for b in BENCHES}

rows = []
for (bench, c) in runs:
    path = RESULTS / "raw" / f"{bench}_{c}_{run_id(bench, c)}.jsonl"
    rows += [{**json.loads(line), "bench": bench} for line in path.read_text().splitlines()]
df = pd.DataFrame(rows)
df["correct"] = [bool(s.get(HEADLINE[b])) for s, b in zip(df.score, df.bench)]
df["muse_tokens"] = df.muse_prompt_tokens + df.muse_completion_tokens
df["muse_usd"] = df.muse_prompt_tokens * PRICE_IN + df.muse_completion_tokens * PRICE_OUT
df["escalated"] = df.route.str.startswith("escalate") | (df.route == "muse_direct")
category = {item.id: CATEGORIES.get(item.meta["category"]) for item in locomo.load(N, SEED)}
df["category"] = df.item_id.map(category)


def wide(bench: str, column: str = "correct") -> pd.DataFrame:
    # One row per item answered by every condition, one column per condition.
    sub = df[df.bench == bench].pivot(index="item_id", columns="baseline", values=column)
    return sub[conditions[bench]].dropna()

## 1. What was run

One row per condition. Every result must come from the real models (not simulated, not a stand-in), at answer temperature 0, with the concurrent ingest validated by the 100-item check.

In [ ]:
KEYS = ["items", "llama_model", "muse_model", "simulated", "muse_is_standin", "answer_temperature",
        "ingest_concurrency", "offline_replay"]
provenance = pd.DataFrame([{"bench": b, "condition": c, **{k: runs[b, c].get(k) for k in KEYS}}
                           for b in BENCHES for c in conditions[b]])
real = not provenance.simulated.any() and not provenance.muse_is_standin.any()
common = {b: len(wide(b)) for b in BENCHES}
display(Markdown(
    f"**{'REAL models' if real else 'NOT the real models: these are not results'}.** "
    + ", ".join(f"{b}: {common[b]} items answered by all {len(conditions[b])} conditions" for b in BENCHES)
    + ". `offline_replay` marks the learned gate, scored with each route's recorded answers."))
provenance

## 2. The north star, per benchmark

- **gap**: accuracy minus `muse_direct`'s, in points. The 95% interval comes from a paired bootstrap over items, so every resample uses the same items for every condition.
- **token share**: Muse tokens (prompt + completion) as a share of `muse_direct`'s on the same items. **input share** counts prompt tokens only. Muse is a reasoning model, so every escalation costs ~1,000–3,000 output tokens however short its prompt. The input share shows how much context the gateway saved.
- **verdict**: the pass rule on the point estimates. It's flagged when the interval still crosses −3. **passes on input** applies the same rule with the input share.

In [ ]:
rng = np.random.default_rng(0)
B = 4000
rows = []
for bench in BENCHES:
    correct = wide(bench).astype(float)
    idx = rng.integers(0, len(correct), size=(B, len(correct)))
    boot = {c: correct[c].to_numpy()[idx].mean(axis=1) * 100 for c in correct}
    sub = df[(df.bench == bench) & df.item_id.isin(correct.index)]
    tokens = sub.groupby("baseline")[["muse_prompt_tokens", "muse_tokens"]].sum()
    for c in conditions[bench]:
        gap = boot[c] - boot["muse_direct"]
        rows.append({
            "bench": bench, "condition": c, "accuracy": correct[c].mean() * 100,
            "acc_lo": np.percentile(boot[c], 2.5), "acc_hi": np.percentile(boot[c], 97.5),
            "gap": (correct[c].mean() - correct["muse_direct"].mean()) * 100,
            "gap_lo": np.percentile(gap, 2.5), "gap_hi": np.percentile(gap, 97.5),
            "token_share": tokens.muse_tokens[c] / tokens.muse_tokens["muse_direct"],
            "input_share": tokens.muse_prompt_tokens[c] / tokens.muse_prompt_tokens["muse_direct"]})
star = pd.DataFrame(rows).set_index(["bench", "condition"])


def verdict(r, share="token_share"):
    if r.name[1] == "muse_direct":
        return "reference"
    failed = [why for why, bad in (("accuracy", r.gap < -MARGIN), ("tokens", r[share] > BUDGET)) if bad]
    if failed:
        return "FAIL: " + " and ".join(failed)
    return "PASS" if r.gap_lo >= -MARGIN else "PASS (interval crosses −3)"


star["verdict"] = star.apply(verdict, axis=1)
star["passes on input"] = star.apply(lambda r: verdict(r, "input_share").startswith("PASS")
                                     if r.name[1] != "muse_direct" else None, axis=1)
star.style.format({"accuracy": "{:.1f}", "acc_lo": "{:.1f}", "acc_hi": "{:.1f}", "gap": "{:+.1f}",
                   "gap_lo": "{:+.1f}", "gap_hi": "{:+.1f}", "token_share": "{:.1%}",
                   "input_share": "{:.1%}"}, na_rep="")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7.2), gridspec_kw={"width_ratios": [1.15, 1]})
for (a1, a2), bench in zip(axes, BENCHES):
    t = star.loc[bench].drop("muse_direct")
    ys = np.arange(len(t))
    a1.axvspan(min(-MARGIN, t.gap_lo.min()) - 5, -MARGIN, color=GRID, alpha=0.6, lw=0)
    a1.axvline(0, color=AXIS, lw=1)
    a1.axvline(-MARGIN, color=MUTED, lw=1, ls="--")
    a1.errorbar(t.gap, ys, xerr=[t.gap - t.gap_lo, t.gap_hi - t.gap], fmt="o", color=BLUE,
                ecolor=BLUE, elinewidth=2, capsize=0, ms=8)
    for y, (c, r) in zip(ys, t.iterrows()):
        a1.annotate(f"{r.gap:+.1f}", (r.gap, y), xytext=(0, 8), textcoords="offset points",
                    ha="center", fontsize=9, color=INK_2)
    a1.set_yticks(ys, t.index)
    a1.invert_yaxis()
    a1.set_ylim(len(t) - 0.4, -0.7)
    a1.grid(axis="y", visible=False)
    ref = star.loc[(bench, "muse_direct"), "accuracy"]
    a1.set_title(f"{bench}: accuracy vs muse_direct ({ref:.1f}%), points", loc="left", fontsize=11)

    h = 0.36
    a2.barh(ys - h / 2, t.token_share * 100, height=h, color=BLUE, label="all Muse tokens")
    a2.barh(ys + h / 2, t.input_share * 100, height=h, color=ORANGE, label="prompt tokens only")
    for y, (c, r) in zip(ys, t.iterrows()):
        a2.text(r.token_share * 100, y - h / 2, f" {r.token_share:.1%}", va="center", fontsize=8, color=INK_2)
        a2.text(r.input_share * 100, y + h / 2, f" {r.input_share:.1%}", va="center", fontsize=8, color=INK_2)
    a2.axvline(BUDGET * 100, color=MUTED, lw=1, ls="--")
    a2.set_yticks(ys, [""] * len(t))
    a2.invert_yaxis()
    a2.set_xlim(0, max(30, t.token_share.max() * 120))
    a2.grid(axis="y", visible=False)
    a2.set_title("Muse tokens, % of muse_direct", loc="left", fontsize=11)
axes[0, 1].legend(frameon=False, fontsize=8, loc="lower right")
for ax in axes[1]:
    ax.set_xlabel("dashed line: the north star's limit")
plt.tight_layout()
plt.show()

### Runaway answers

Now and then Muse reasons until it hits its output limit (262,144 tokens) and returns no answer. Each costs about $1.10 and is scored wrong. That is a real cost of calling Muse, so the shares above keep it. But it sits in `muse_direct`, the denominator, so it makes every gateway share look smaller. Below are the shares with those items dropped from every condition.

In [ ]:
RUNAWAY = 262_144  # Muse's output limit: it reasoned until it was cut off
runaway = df[df.muse_completion_tokens >= RUNAWAY]
display(runaway[["bench", "baseline", "item_id", "muse_completion_tokens", "correct", "answer"]])
rows, runaway_share = [], {}
for b in BENCHES:
    tokens = wide(b, "muse_tokens")
    dropped = tokens.index.isin(set(runaway[runaway.bench == b].item_id))
    runaway_share[b] = tokens.muse_direct[dropped].sum() / tokens.muse_direct.sum()
    for c in conditions[b]:
        if c != "muse_direct":
            rows.append({"bench": b, "condition": c, "token_share": star.loc[(b, c), "token_share"],
                         "runaway items dropped": tokens[c][~dropped].sum() / tokens.muse_direct[~dropped].sum()})
display(pd.DataFrame(rows).set_index(["bench", "condition"]).style.format("{:.1%}"))
display(Markdown(", ".join(f"{b}: runaways are {runaway_share[b]:.1%} of `muse_direct`'s tokens" for b in BENCHES) + "."))

## 3. Cost and latency

Per question. Dollars use the Muse prices in `.env`. Seconds are wall time per question as the harness saw it, including the round trip to Modal or the Meta API. **cached share** is the share of Llama's prompt tokens served from the prefix cache.

In [ ]:
g = df.groupby(["bench", "baseline"])
cost = pd.DataFrame({
    "accuracy": g.correct.mean() * 100,
    "escalated": g.escalated.mean() * 100,
    "muse_prompt": g.muse_prompt_tokens.mean(),
    "muse_completion": g.muse_completion_tokens.mean(),
    "usd_per_1k_questions": g.muse_usd.mean() * 1000,
    "median_s": g.seconds.median(),
    "p90_s": g.seconds.quantile(0.9),
    "llama_compute": g.compute_tokens.mean(),
    "cached_share": g.cached_tokens.sum() / g.llama_prompt_tokens.sum().replace(0, np.nan),
}).reindex([(b, c) for b in BENCHES for c in conditions[b]])
cost.style.format({"accuracy": "{:.1f}", "escalated": "{:.0f}%", "muse_prompt": "{:,.0f}",
                   "muse_completion": "{:,.0f}", "usd_per_1k_questions": "${:,.2f}",
                   "median_s": "{:.1f}", "p90_s": "{:.1f}", "llama_compute": "{:,.0f}",
                   "cached_share": "{:.0%}"}, na_rep="")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 0.45 * max(map(len, conditions.values())) + 1.6))
for ax, bench in zip(axes, BENCHES):
    t = cost.loc[bench]
    ys = np.arange(len(t))
    ax.barh(ys, t.muse_prompt, color=BLUE, height=0.55, label="prompt")
    ax.barh(ys, t.muse_completion, left=t.muse_prompt, color=ORANGE, height=0.55,
            edgecolor=SURFACE, linewidth=2, label="completion (incl. reasoning)")
    for y, (c, r) in zip(ys, t.iterrows()):
        ax.text(r.muse_prompt + r.muse_completion, y, f" {r.muse_prompt + r.muse_completion:,.0f}",
                va="center", fontsize=8, color=INK_2)
    ax.set_yticks(ys, t.index)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlim(0, (t.muse_prompt + t.muse_completion).max() * 1.2)
    ax.set_title(f"{bench}: Muse tokens per question", loc="left", fontsize=11)
axes[0].legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

## 4. LoCoMo: where the gap comes from

LoCoMo is the benchmark where the gateway is furthest from `muse_direct`. Two things can lose an item:
- **Retrieval:** none of the top-5 memories comes from a session that holds the evidence (`source_in_hits`).
- **Compression or model:** the right session was retrieved, but its summary dropped the detail, or the model got it wrong.

`muse_direct` reads the whole conversation, so it is affected by neither.

In [ ]:
lo = df[(df.bench == "locomo") & df.item_id.isin(wide("locomo").index)]
by_category = lo.pivot_table(index="category", columns="baseline", values="correct",
                             aggfunc="mean").reindex(CATEGORIES.values())[conditions["locomo"]] * 100
by_category.insert(0, "items", lo[lo.baseline == "muse_direct"].groupby("category").size())
by_category.loc["all"] = [len(wide("locomo")), *(lo.groupby("baseline").correct.mean()[conditions["locomo"]] * 100)]
display(Markdown("**Accuracy (%) by question type**"))
display(by_category.style.format("{:.0f}").background_gradient(
    cmap="Blues", vmin=0, vmax=100, subset=conditions["locomo"]))

In [ ]:
retrieved = lo[lo.baseline == "llama_only"].set_index("item_id").source_in_hits.astype(bool)
memory_conditions = [c for c in conditions["locomo"] if c != "muse_direct"]
correct = wide("locomo").astype(bool)
split = pd.DataFrame({
    c: {"source retrieved: right": (correct[c] & retrieved).sum(),
        "source retrieved: wrong": (~correct[c] & retrieved).sum(),
        "not retrieved: right": (correct[c] & ~retrieved).sum(),
        "not retrieved: wrong": (~correct[c] & ~retrieved).sum()}
    for c in conditions["locomo"]}).T
segments = [("source retrieved: right", BLUE, None), ("source retrieved: wrong", ORANGE, None),
            ("not retrieved: right", BLUE, "////"), ("not retrieved: wrong", ORANGE, "////")]
fig, ax = plt.subplots(figsize=(10, 0.5 * len(split) + 1.4))
left = np.zeros(len(split))
for name, colour, hatch in segments:
    ax.barh(split.index, split[name], left=left, color=colour, hatch=hatch, height=0.55,
            edgecolor=SURFACE, linewidth=2, label=name)
    for y, (v, x) in enumerate(zip(split[name], left)):
        if v >= 12:
            ax.text(x + v / 2, y, str(v), ha="center", va="center", fontsize=8, color="white")
    left += split[name].to_numpy()
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("questions")
ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.22))
ax.set_title("LoCoMo: right or wrong, by whether memory from the evidence's session was retrieved",
             loc="left", fontsize=11)
plt.tight_layout()
plt.show()

rates = pd.DataFrame({c: {"accuracy, source retrieved": correct[c][retrieved].mean() * 100,
                          "accuracy, not retrieved": correct[c][~retrieved].mean() * 100}
                      for c in conditions["locomo"]}).T
display(Markdown(
    f"Retrieval finds the evidence's session for **{retrieved.mean():.0%}** of questions. "
    "The same item split applies to `muse_direct`, which shows how hard each subset is "
    "when nothing is lost."))
rates.round(1)

## 5. SWE-bench: what its accuracy measures

The task is to name the file to change. Its "evidence" is the gold file:
- **`muse_direct`:** the gold file is in view if it is among the ~11K tokens of code retrieved for the full context (`in_context`, the ceiling).
- **Gateway conditions:** it is in view if one of the top-5 memories comes from it (`gold_in_hits`).

When the gold file isn't in view, a right answer comes from the issue text (which often names the file or module) or from the model's knowledge of the repo.

In [ ]:
sw = df[(df.bench == "swebench") & df.item_id.isin(wide("swebench").index)].copy()
sw["evidence shown"] = np.where(sw.baseline == "muse_direct", sw.in_context,
                                sw.gold_in_hits.fillna(False)).astype(bool)
shown = sw.groupby(["baseline", "evidence shown"]).correct.agg(["mean", "size"]).unstack()
shown.columns = [f"{'accuracy' if m == 'mean' else 'items'}, {'shown' if s else 'not shown'}"
                 for m, s in shown.columns]
shown = shown.reindex(conditions["swebench"])
shown[[c for c in shown if c.startswith("accuracy")]] *= 100
display(shown.style.format("{:.0f}"))
md_ = shown.loc["muse_direct"]
display(Markdown(
    f"`muse_direct` names the right file for **{md_['accuracy, not shown']:.0f}%** of the "
    f"{md_['items, not shown']:.0f} items whose gold file wasn't in its context. "
    "Muse's knowledge of these repos (and the issue text) carries much of SWE-bench's accuracy. "
    "That is why Muse with only the memory summaries can match it, and why LoCoMo, "
    "where the answer exists only in the team's conversation, is the stricter test of memory."))

## 6. Routing headroom: the best any gate could do

A gate only chooses between answers that already exist: Llama's draft (`llama_only`), or an escalation. The **oracle** knows which ones are right and picks the cheapest right one: the draft if it's right, else the compact escalation, else (if it was run) the raw one. If none is right, it keeps the draft. No gate can beat it with these routes. A gate's accuracy can only reach `muse_direct`'s if the oracle's does.

In [ ]:
def oracle(bench: str, routes: list[str]) -> dict:
    correct, tokens = wide(bench).astype(bool), wide(bench, "muse_tokens")
    picked = pd.Series(routes[0], index=correct.index)
    for route in reversed(routes):  # the cheapest right route wins
        picked[correct[route]] = route
    acc = np.mean([correct.at[i, r] for i, r in picked.items()]) * 100
    spent = sum(tokens.at[i, r] for i, r in picked.items())
    return {"accuracy": acc, "token_share": spent / tokens.muse_direct.sum(),
            "escalated": (picked != routes[0]).mean() * 100}


rows = []
for bench in BENCHES:
    for c in conditions[bench]:
        r = star.loc[bench, c]
        rows.append({"bench": bench, "condition": c, "accuracy": r.accuracy,
                     "token_share": r.token_share, "escalated": cost.loc[(bench, c), "escalated"]})
    routes = ["llama_only", "escalate_compact"]
    rows.append({"bench": bench, "condition": "oracle: draft or compact", **oracle(bench, routes)})
    if "escalate_raw" in conditions[bench]:
        rows.append({"bench": bench, "condition": "oracle: draft, compact or raw",
                     **oracle(bench, routes + ["escalate_raw"])})
headroom = pd.DataFrame(rows).set_index(["bench", "condition"])
headroom.style.format({"accuracy": "{:.1f}", "token_share": "{:.1%}", "escalated": "{:.0f}%"})

In [ ]:
rows = []
for bench in BENCHES:
    correct = wide(bench).astype(bool)
    both = pd.crosstab(correct.llama_only.map({True: "draft right", False: "draft wrong"}),
                       correct.escalate_compact.map({True: "compact right", False: "compact wrong"}))
    display(Markdown(f"**{bench}**: Llama's draft vs the compact escalation, items"))
    display(both)
    lost = correct.muse_direct & ~correct[[c for c in correct if c != "muse_direct"]].any(axis=1)
    display(Markdown(f"{lost.sum()} {bench} items are right only with `muse_direct`: "
                     "no gateway route answers them."))

In [ ]:
gate_file = RESULTS / f"gate_{RUN_ID}.json"
gate = json.loads(gate_file.read_text()) if gate_file.exists() else None
markers = {"muse_direct": "s", "oracle: draft or compact": "D", "oracle: draft, compact or raw": "D"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, bench in zip(axes, BENCHES):
    t = headroom.loc[bench]
    ref = t.accuracy["muse_direct"]
    ax.axhspan(ref - MARGIN, 100, color=GRID, alpha=0.5, lw=0)
    ax.axvline(BUDGET * 100, color=MUTED, lw=1, ls="--")
    if gate:
        sweep = pd.DataFrame([{"share": p[bench]["muse_token_share"] * 100,
                               "accuracy": p[bench]["accuracy"] * 100} for p in gate["sweep"]])
        sweep = sweep.sort_values("share")
        ax.plot(sweep.share, sweep.accuracy, color=AQUA, lw=2, label="per-route gate, λ sweep", zorder=2)
    # Conditions at the same point (e.g. a gate that never escalates) share one label.
    for (share, acc), names in t.groupby([t.token_share.round(4), t.accuracy.round(2)]).groups.items():
        hollow = names[0].startswith("oracle")
        ax.scatter(share * 100, acc, s=70, marker=markers.get(names[0], "o"), zorder=3,
                   facecolor=SURFACE if hollow else BLUE, edgecolor=BLUE, linewidth=2)
        left = share > 0.5  # label the right-most points below on their left
        ax.annotate(", ".join(names), (share * 100, acc), xytext=(-8 if left else 7, -14 if left else -3),
                    textcoords="offset points", ha="right" if left else "left", fontsize=8, color=INK_2)
    top = max(t.accuracy.max(), ref) + 3
    ax.set_ylim(min(t.accuracy.min(), ref - MARGIN) - 3, top)
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xlim(-0.3, 400)
    ax.set_xticks([0, 1, 10, 100], ["0%", "1%", "10%", "100%"])
    ax.set_xlabel("Muse tokens, % of muse_direct (log above 1%)")
    ax.set_ylabel("accuracy (%)")
    ax.set_title(f"{bench}: cost vs accuracy", loc="left", fontsize=11)
    if gate:
        ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.text(0.01, -0.02, "Grey band: within 3 points of muse_direct. Dashed line: 10% of its tokens. "
         "The north star is the band's part left of the line. Hollow diamonds: the oracle.",
         fontsize=8, color=MUTED)
plt.tight_layout()
plt.show()

In [ ]:
# Each gate's choices against the oracle. Routes from cheapest to dearest; a gate is only
# judged on the routes it could take.
ROUTE_CONDITION = {"local_memory": "llama_only", "escalate_compact": "escalate_compact",
                   "escalate_raw": "escalate_raw"}
OUTCOMES = ["right: the cheapest right route", "right, but a cheaper route was also right",
            "wrong: a route it didn't take was right", "wrong: no route was right"]
for bench in BENCHES:
    correct = wide(bench).astype(bool)
    chosen = wide(bench, "route")
    table = {}
    for g in [c for c in ("rule_gate", "full") if c in conditions[bench]]:
        routes = ["local_memory", "escalate_compact"] + (["escalate_raw"] if g == "full" and
                                                         "escalate_raw" in conditions[bench] else [])
        outcome = []
        for item, route in chosen[g].items():
            right = [r for r in routes if correct.at[item, ROUTE_CONDITION[r]]]
            if route in right:
                outcome.append(OUTCOMES[0] if route == right[0] else OUTCOMES[1])
            else:
                outcome.append(OUTCOMES[2] if right else OUTCOMES[3])
        table[g] = pd.Series(outcome).value_counts()
    display(Markdown(f"**{bench}: each gate's decisions against the oracle, items**"))
    display(pd.DataFrame(table).reindex(OUTCOMES).fillna(0).astype(int))

## 7. Team scaling (replay)

SWE-bench tasks on one repo, done by 1–16 simulated engineers, two tasks each. Per-engineer cost should fall as the team grows only if the team really shares work:
- **Shared** (blue): one team, with shared memory and cache.
- **No sharing** (orange): every engineer on their own.
- **Solid** lines are related tasks (a team working in one area); **dashed** lines are random tasks.

Pages ingested and compute tokens don't depend on timing. Seconds do: ingest runs 16 calls at a time and the GPU starts cold, so they are noisy.

In [ ]:
csvs = sorted(RESULTS.glob("replay_*.csv"))
replay = pd.concat([pd.read_csv(p).assign(file=p.name) for p in csvs])
replay = replay[~replay.simulated.astype(bool) & ~replay.muse_is_standin.astype(bool)]
# The latest real-model replay per (baseline, order): file names end in the run's timestamp.
replay = replay[replay.file == replay.groupby(["baseline", "order"]).file.transform("max")]
metrics = [("ingest_pages_per_user", "memory pages ingested / engineer"),
           ("compute_tokens_per_user", "Llama compute tokens / engineer"),
           ("muse_tokens_per_user", "Muse tokens / engineer"),
           ("ingest_seconds_per_user", "ingest seconds / engineer")]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, (metric, title) in zip(axes, metrics):
    for (baseline, order), sub in replay.groupby(["baseline", "order"]):
        sub = sub.sort_values("team_size")
        ax.plot(sub.team_size, sub[metric], color=ORANGE if baseline == "no_sharing" else BLUE,
                lw=2, marker="o", ms=5, ls="--" if order == "random" else "-",
                label=f"{'no sharing' if baseline == 'no_sharing' else 'shared'}, {order}")
    ax.set_xscale("log", base=2)
    ax.set_xticks(sorted(replay.team_size.unique()))
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xlabel("team size")
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_ylim(bottom=0)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

ends = replay.pivot_table(index=["order", "baseline"], columns="team_size",
                          values="ingest_pages_per_user")
ends["change, 1 → 16"] = ends[16] / ends[1] - 1
display(Markdown("**Pages ingested per engineer**"))
display(ends.style.format("{:.1f}").format("{:+.0%}", subset=["change, 1 → 16"]))
display(Markdown(f"Files: {', '.join(sorted(replay.file.unique()))}"))

## 8. The learned gate

Written by `scripts/gate_run.sh` (`eval/confidence.py`, then `eval/learn_gate.py`).
- **The model:** one logistic model per route. Its features come from the question, the retrieved memory and Llama's draft. When `eval/confidence.py` has run, they also include Llama's confidence in its draft:
  - the probabilities it gave the draft's tokens;
  - whether 4 resampled answers agree with the draft;
  - its P(yes) when asked whether the draft is right.
- **Scored out of fold:** whole teams are held out, so every number is on conversations and repos the gate didn't train on.
- **λ:** for each λ the gate picks the route with the best P(right) − λ × estimated Muse dollars. Sweeping λ draws the curve.
- **Operating points:** the verdict for each benchmark uses the most accurate λ within *that benchmark's* 10%, because a team sets λ against its own budget. Choosing one number from the sweep is a mild selection on the same items. The pooled λ, one value for both benchmarks, is shown for reference; it is the live gateway's default.
- **The signal table:** how well each feature, and the gate's out-of-fold P(draft right), separate right drafts from wrong ones. It is reported as AUC: 0.5 is chance and 1 is perfect. A gate can only beat random escalation if this is well above 0.5.

In [ ]:
if gate is not None and "signal" in gate:
    sig = pd.DataFrame({b: {"gate: P(draft right)": gate["signal"][b]["gate_auc"],
                            **gate["signal"][b]["feature_auc"]} for b in BENCHES if b in gate["signal"]})
    display(Markdown(f"**AUC for telling a right Llama draft from a wrong one** "
                     f"({'with' if gate.get('confidence') else 'without'} Llama's confidence). "
                     "Below 0.5: higher values mean *wrong*, which is just as useful."))
    display(sig.style.format("{:.2f}", na_rep="").background_gradient(cmap="Blues", vmin=0.5, vmax=1.0))

### The cascade: the gate the gateway ships

The per-route gate above barely beats random escalation on code, though its signal is decent. The **cascade** (`gateway.gate.CascadeGate`) is simpler:
- **One score:** a single logistic model scores P(draft right) from Llama's six confidence features only. The feature set was fixed before scoring.
- **One rule:** drafts under a threshold escalate. Code memory escalates with its summaries, any other memory with the raw pages (the routes each benchmark favoured).
- **Out of fold:** the same team folds as the per-route gate, so the two are comparable. Sweeping the threshold draws its curve.
- **`full`:** the `full` condition in every table is the cascade at each benchmark's 10%.

Two more reference points:
- **A perfect draft check:** it escalates exactly the wrong drafts. It shows how far a gate that only judges the draft could go. The oracle, by contrast, also knows when Muse will fail.
- **Escalation costs:** Muse writes longer reasoning on the questions Llama gets wrong. Those are the questions a good gate escalates, so a gate's escalations cost more than average.

In [ ]:
BEST_ROUTE = {"swebench": "escalate_compact", "locomo": "escalate_raw"}


def random_escalation(b, share):
    # Escalate a random fraction of questions down one route (in expectation), keep the
    # draft for the rest; the best such route at this token share. A gate must beat it.
    local = star.loc[(b, "llama_only")]
    best = local.accuracy
    for c in ("escalate_compact", "escalate_raw"):
        if c in conditions[b]:
            r = star.loc[(b, c)]
            best = max(best, local.accuracy + (r.accuracy - local.accuracy) * min(1, share / r.token_share))
    return best


cascade, cascade_rows = {}, []
for b in BENCHES:
    if gate is None or "cascade" not in gate or BEST_ROUTE[b] not in conditions[b]:
        continue
    usd_direct = wide(b, "muse_usd").muse_direct.sum()
    cascade[b] = pd.DataFrame([{"threshold": p["threshold"], "token_share": p[b]["muse_token_share"],
                                "input_share": p[b]["muse_input_share"], "usd_share": p[b]["muse_usd"] / usd_direct,
                                "accuracy": p[b]["accuracy"] * 100, "kept": p[b]["routes"].get("local_memory", 0.0)}
                               for p in gate["cascade"]["sweep"]])
    correct, tokens = wide(b).astype(bool), wide(b, "muse_tokens")
    draft, escalation, spent = correct.llama_only, correct[BEST_ROUTE[b]], tokens[BEST_ROUTE[b]]
    own = gate["cascade"]["operating_points"][f"{BUDGET:g}"][b]
    per_route = gate["operating_points"][b]
    o = headroom.loc[(b, [n for n in headroom.loc[b].index if n.startswith("oracle")][-1])]
    cascade_rows += [
        {"bench": b, "gate": "per-route gate (all features), its own 10%",
         "accuracy": per_route["accuracy"] * 100, "token_share": per_route["muse_token_share"]},
        {"bench": b, "gate": "cascade (confidence only), its own 10%",
         "accuracy": own["accuracy"] * 100, "token_share": own["muse_token_share"]},
        {"bench": b, "gate": "perfect draft check",
         "accuracy": np.where(~draft, escalation, draft).mean() * 100,
         "token_share": spent[~draft].sum() / tokens.muse_direct.sum()},
        {"bench": b, "gate": "oracle", "accuracy": o.accuracy, "token_share": o.token_share}]
    display(Markdown(
        f"**{b}**: `{BEST_ROUTE[b]}` costs **{spent[~draft].mean():,.0f}** Muse tokens per question when "
        f"Llama's draft is wrong, and {spent[draft].mean():,.0f} when it is right. It fixes "
        f"{escalation[~draft].mean():.0%} of the wrong drafts and keeps {escalation[draft].mean():.0%} of the right ones."))

if cascade_rows:
    display(Markdown(f"Cascade AUC for telling a right draft from a wrong one: "
                     + ", ".join(f"{b} {gate['cascade']['auc'][b]:.2f}" for b in BENCHES if b in cascade) + "."))
    gates = pd.DataFrame(cascade_rows).set_index(["bench", "gate"])
    gates["random, same tokens"] = [random_escalation(b, r.token_share) for (b, _), r in gates.iterrows()]
    gates["gap to muse_direct"] = [r.accuracy - star.loc[(b, "muse_direct"), "accuracy"] for (b, _), r in gates.iterrows()]
    display(gates.style.format({"accuracy": "{:.1f}", "token_share": "{:.1%}", "random, same tokens": "{:.1f}",
                                "gap to muse_direct": "{:+.1f}"}, na_rep=""))

In [ ]:
if gate is None:
    display(Markdown(f"**Not run yet.** After `scripts/gate_run.sh`, `{gate_file.name}` appears; rerun "
                     "the notebook. The `full` condition then shows up in every table above."))
else:
    point = gate["operating_point"]
    sweep = pd.DataFrame([{"lam": p["lam"], "bench": b, "accuracy": p[b]["accuracy"] * 100,
                           "token_share": p[b]["muse_token_share"], "input_share": p[b]["muse_input_share"],
                           **{f"{r}": p[b]["routes"].get(r, 0.0) for r in gate["routes"]}}
                          for p in gate["sweep"] for b in BENCHES if b in p])
    ref = {b: star.loc[(b, "muse_direct"), "accuracy"] for b in BENCHES}

    rows = []
    for b in BENCHES:
        s = sweep[sweep.bench == b]
        pooled = s[s.lam == point["lam"]].iloc[0]
        own = s[s.token_share <= BUDGET].sort_values(["accuracy", "lam"], ascending=False).iloc[0]
        for choice, r in (("one λ, pooled 10%", pooled), ("λ per benchmark, its own 10%", own)):
            rows.append({"bench": b, "choice": choice, "λ": r.lam, "accuracy": r.accuracy,
                         "gap": r.accuracy - ref[b], "random, same tokens": random_escalation(b, r.token_share),
                         "token_share": r.token_share,
                         "input_share": r.input_share, **{k: r[k] for k in gate["routes"]},
                         "verdict": "PASS" if r.accuracy - ref[b] >= -MARGIN and r.token_share <= BUDGET
                         else "FAIL"})
    choices = pd.DataFrame(rows).set_index(["bench", "choice"])
    display(Markdown(f"Routes {gate['routes']}, {gate['items']} items; "
                     f"{gate['retrieval_mismatches']} retrievals differ from the run's. "
                     "Route columns: share of questions sent down each route."))
    display(choices.style.format({"λ": "{:.3g}", "accuracy": "{:.1f}", "gap": "{:+.1f}",
                                  "random, same tokens": "{:.1f}",
                                  "token_share": "{:.1%}", "input_share": "{:.1%}",
                                  **{k: "{:.0%}" for k in gate["routes"]}}))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, b in zip(axes, BENCHES):
        s = sweep[sweep.bench == b].sort_values("token_share")
        ax.axhspan(ref[b] - MARGIN, 100, color=GRID, alpha=0.5, lw=0)
        ax.axvline(BUDGET * 100, color=MUTED, lw=1, ls="--")
        ax.plot(s.token_share * 100, s.accuracy, color=AQUA, lw=2, marker="o", ms=3,
                label="per-route gate, λ sweep")
        if b in cascade:
            ax.plot(cascade[b].token_share * 100, cascade[b].accuracy, color=ORANGE, lw=2,
                    label="cascade (confidence only)")
        h = headroom.loc[b]
        local = h.loc["llama_only"]
        for i, c in enumerate(c for c in ("escalate_compact", "escalate_raw") if c in h.index):
            ax.plot([0, h.loc[c].token_share * 100], [local.accuracy, h.loc[c].accuracy], color=MUTED,
                    lw=1, ls=":", label="random escalation" if i == 0 else None)
        best_oracle = [c for c in h.index if c.startswith("oracle")][-1]
        for c in ["llama_only", "rule_gate", "escalate_compact", "escalate_raw", best_oracle]:
            if c in h.index and not (c == "rule_gate" and h.loc[c].equals(h.loc["llama_only"])):
                r = h.loc[c]
                ax.scatter(r.token_share * 100, r.accuracy, s=60, zorder=3, marker="D" if c == best_oracle else "o",
                           facecolor=SURFACE if c == best_oracle else BLUE, edgecolor=BLUE, linewidth=2)
                ax.annotate(c, (r.token_share * 100, r.accuracy), xytext=(6, 4), textcoords="offset points",
                            fontsize=8, color=INK_2)
        o = choices.loc[(b, "λ per benchmark, its own 10%")]
        ax.scatter(o.token_share * 100, o.accuracy, s=90, zorder=4, color=AQUA, edgecolor=SURFACE, linewidth=2)
        ax.annotate("per-route gate at its own 10%", (o.token_share * 100, o.accuracy), xytext=(6, -12),
                    textcoords="offset points", fontsize=8, color=INK_2)
        ax.set_xlim(0, 40)
        ax.set_ylim(min(s.accuracy.min(), ref[b] - MARGIN) - 4, max(ref[b], h.accuracy.max()) + 4)
        ax.set_xlabel("Muse tokens, % of muse_direct")
        ax.set_ylabel("accuracy (%)")
        ax.set_title(f"{b}: the gates vs the oracle (muse_direct {ref[b]:.1f}%)", loc="left", fontsize=11)
    axes[0].legend(frameon=False, fontsize=8, loc="lower right")
    fig.text(0.01, -0.02, "Grey band: within 3 points of muse_direct. Dashed line: 10% of its tokens. "
             "Hollow diamond: the oracle over every route. Dotted: escalating questions at random. "
             "Both gate curves are out of fold.", fontsize=8, color=MUTED)
    plt.tight_layout()
    plt.show()

### Larger budgets: 20% and 25% (chosen after the run)

The north star's limit is 10% of `muse_direct`'s Muse tokens, and its verdict above stands. These budgets were picked **after** seeing the results, on the view that a team would rather pay more for accuracy. Read them as points on the cost/accuracy curve, not as a pass.

For each budget, the most accurate option whose Muse tokens fit, among:
- every fixed condition;
- the per-route gate's λ sweep;
- the cascade's threshold sweep.

**kept on Llama** is the share of questions answered by Llama's draft. **without runaways** divides by `muse_direct`'s tokens minus its runaway answers (see section 2). It is an upper bound: the option's own tokens on those items are kept.

**dollar share** is Muse dollars as a share of `muse_direct`'s, at the prices in `.env`. Muse's output costs more than its input, and escalations are mostly output, so it sits above the token share.

In [ ]:
BUDGETS = [0.10, 0.20, 0.25]
options = []
for b in BENCHES:
    usd = df[(df.bench == b) & df.item_id.isin(wide(b).index)].groupby("baseline").muse_usd.sum()
    for c in conditions[b]:
        if c not in ("muse_direct", "full"):
            r = star.loc[(b, c)]
            label = {"escalate_compact": "escalate_compact: every question, with summaries",
                     "escalate_raw": "escalate_raw: every question, with raw pages"}.get(c, c)
            options.append({"bench": b, "option": label, "accuracy": r.accuracy, "token_share": r.token_share,
                            "input_share": r.input_share, "usd_share": usd[c] / usd["muse_direct"],
                            "kept": 1 - cost.loc[(b, c), "escalated"] / 100})
    if gate is not None:
        for p in gate["sweep"]:
            if b in p:
                q = p[b]
                options.append({"bench": b, "option": f"per-route gate, λ = {p['lam']:.3g}",
                                "accuracy": q["accuracy"] * 100, "token_share": q["muse_token_share"],
                                "input_share": q["muse_input_share"], "usd_share": q["muse_usd"] / usd["muse_direct"],
                                "kept": q["routes"].get("local_memory", 0.0)})
    if b in cascade:
        for r in cascade[b].itertuples():
            options.append({"bench": b, "option": f"cascade, threshold {r.threshold:.2f}", "accuracy": r.accuracy,
                            "token_share": r.token_share, "input_share": r.input_share, "usd_share": r.usd_share,
                            "kept": r.kept})
options = pd.DataFrame(options)

rows = []
for b in BENCHES:
    ref = star.loc[(b, "muse_direct"), "accuracy"]
    for budget in BUDGETS:
        fits = options[(options.bench == b) & (options.token_share <= budget + 1e-9)]
        best = fits.sort_values(["accuracy", "token_share"], ascending=[False, True]).iloc[0]
        rows.append({"bench": b, "budget": f"{budget:.0%}", "most accurate option": best.option,
                     "accuracy": best.accuracy, "gap": best.accuracy - ref, "token_share": best.token_share,
                     "without runaways": best.token_share / (1 - runaway_share[b]),
                     "input_share": best.input_share, "usd_share": best.usd_share, "kept on Llama": best.kept,
                     "within 3 points": "yes" if best.accuracy - ref >= -MARGIN else "no"})
budgets = pd.DataFrame(rows).set_index(["bench", "budget"])
display(budgets.style.format({"accuracy": "{:.1f}", "gap": "{:+.1f}", "token_share": "{:.1%}",
                              "without runaways": "{:.1%}", "input_share": "{:.1%}", "usd_share": "{:.0%}",
                              "kept on Llama": "{:.0%}"}))
display(Markdown(
    "The 10% rows are the north star (all routes and gates scored out of fold or as run). "
    "A cascade or gate point is one threshold on 300 questions; neighbouring points differ by one question."))

## 9. Summary

Generated from the numbers above, so it stays current if anything is rerun.

In [ ]:
lines = []
for bench in BENCHES:
    t = star.loc[bench]
    ref = t.accuracy["muse_direct"]
    best = t.drop("muse_direct").sort_values(["gap", "token_share"], ascending=[False, True]).iloc[0]
    cheap = t[(t.index != "muse_direct") & (t.token_share <= BUDGET)]
    best_cheap = cheap.sort_values("gap", ascending=False).iloc[0] if len(cheap) else None
    oracle_name = [c for c in headroom.loc[bench].index if c.startswith("oracle")][-1]
    o = headroom.loc[(bench, oracle_name)]
    lines.append(
        f"- **{bench}**: `muse_direct` {ref:.1f}%. Most accurate gateway condition: "
        f"`{best.name}` {best.accuracy:.1f}% ({best.gap:+.1f} pts) at {best.token_share:.1%} of its tokens "
        f"({best.input_share:.1%} of its prompt tokens): {best.verdict}."
        + (f" Best within 10%: `{best_cheap.name}` {best_cheap.accuracy:.1f}% ({best_cheap.gap:+.1f} pts)."
           if best_cheap is not None else "")
        + f" The best oracle (`{oracle_name}`) reaches {o.accuracy:.1f}% at {o.token_share:.1%}.")
display(Markdown("\n".join(lines)))

### Reading these results (31 Oct, after adding Llama's confidence to the gate)

Section 9's summary is generated from the data and stays current. This text was written on 31 Oct, from the full run and the gate with Llama's confidence (`scripts/gate_run.sh`).

**The north star is not met on either benchmark, and a better gate alone can't meet it.**

**1. On code, memory lets Muse match full context with under a tenth of the context.**
- **Results:** with the top-5 memory summaries, Muse scores 86.3% against `muse_direct`'s 86.0%, using 8.9% of its prompt tokens. With the raw pages behind them, it scores 87.0%, using 26%.
- **Why it still fails the 10% limit:** both fail on *total* tokens (23% and 34%), because Muse writes ~2,800 reasoning tokens per escalation however short the prompt.
- **Caveat:** `muse_direct` names the right file for 80% of questions whose gold file it never saw. Much of SWE-bench's accuracy is Muse's knowledge of these repos.

**2. On conversations, raw pages fix the summaries; retrieval is what's left.**
- **Raw vs summaries:** when retrieval finds the evidence's session, Muse with the raw pages scores **85.3%**, close to `muse_direct`'s 88.9%. With summaries it scores 60.5%. Overall, `escalate_raw` lifts LoCoMo from 46.0% to 62.7%, at 15.7% of `muse_direct`'s tokens.
- **The remaining loss is retrieval:** the top-5 memories miss the evidence's session for 37% of questions, and on those every gateway route scores 15–24%. No choice of route reaches more than 66.7% (the oracle), so no gate can pass LoCoMo with this retrieval.

**3. Llama's confidence is a real signal.** AUC for telling a right draft from a wrong one, out of fold:

| | Without confidence | With confidence |
|---|---|---|
| SWE-bench | 0.61 | **0.68** |
| LoCoMo | 0.69 | **0.80** |

- **The strongest features:** whether 4 resampled answers agree with the draft (0.68 on SWE-bench, 0.71 on LoCoMo), and the draft's mean token log-probability (0.60, 0.74).
- **Asking Llama whether its own answer is right:** it helps within each benchmark (0.61, 0.66). Its scale differs between the two, so pooled it is at chance (0.51).
- **Cost:** the probes cost ~200 (SWE-bench) and ~85 (LoCoMo) Llama compute tokens per question, mostly from the prefix cache, and no Muse.

**4. The per-route gate wastes most of that signal on code. The cascade, now the gateway's gate, uses it.** At 10% of Muse tokens, out of fold:

| Benchmark | Gate | Accuracy | Random escalation, same tokens |
|---|---|---|---|
| SWE-bench | per-route gate, all 18 features | 68.0% at 8.9% | 67.5% |
| SWE-bench | cascade, confidence only | **74.0%** at 10.0% | 69.0% |
| LoCoMo | per-route gate | 55.0% at 9.9% | 53.7% |
| LoCoMo | cascade | 54.7% at 9.4% | 53.0% |

- **What the cascade is (`gateway.gate.CascadeGate`):**
  - One logistic model scores P(draft right) from the six confidence features.
  - Drafts under a threshold go to Muse: code memory with summaries, other memory with raw pages.
  - On SWE-bench its curve stays above the per-route gate's all the way to ~20% of the tokens. It is the `full` condition in every table.
- **Selection:** its feature set was fixed before scoring. A per-benchmark version, tried first in this notebook, gave 74.3%. The one other feature set tried (agreement and log-probability only) gave 73.3%.
- **Likely reasons the per-route gate falls behind:**
  - It fits 18 features for each of 3 routes on ~480 training questions per fold, and decides on the difference of two noisy probabilities.
  - It prices every escalation at the average output length. But Muse writes more on the questions Llama gets wrong: 4,543 tokens per escalation against 3,256. Those are exactly the questions a good gate escalates.
- **On LoCoMo, ranking barely matters:** only 38% of drafts are right, and a 10% budget already escalates ~60% of questions.

**5. Why the 10% limit is out of reach for any gate that only judges the draft.**
- **Code:**
  - Escalating exactly the wrong drafts would score 88.3%, but costs 12.1% of the tokens.
  - Within 10%, a perfect draft check reaches about 83%, which is exactly the pass mark.
  - Only the oracle passes (90.3% at 8.8%), because it also knows when Muse will fail.
  - The realistic gates here reach 68–74%.
- **Conversations:** retrieval caps every route at 66.7% (finding 2).

**6. With a larger budget, code reaches `muse_direct`; conversations don't.** These budgets were chosen after the run (section 8, "Larger budgets"):

| Budget | SWE-bench | LoCoMo |
|---|---|---|
| 10% (the north star) | 74.0% (−12.0) | 55.0% (−32.0) |
| 20% | 82.7% (−3.3, one question short) | 62.7% (−24.3) |
| 25% | **86.3% (+0.3)** at 23.0% of the tokens, 36% of the dollars | 62.7% (−24.3) |

- **Code at 25%:** sending every question with summaries matches `muse_direct`. At 20%, the cascade keeps only 16% of Llama's drafts. **With an accuracy-first budget, most of the saving comes from the compact memory, not from answering locally.**
- **Runaway answers:** two of `muse_direct`'s SWE-bench answers reasoned to Muse's 262,144-token output limit and returned nothing (section 2). They are 11% of its tokens. Without them, the 20% and 25% points cost 22.4% and 25.8%. Say "about a quarter of the tokens", not "23%".
- **Conversations:** LoCoMo stops at 62.7% (every question with raw pages, 15.7%) whatever the budget. Only better retrieval moves it.

**Team scaling holds.**
- **Result:** with shared memory, pages ingested per engineer (the gateway's main GPU work) fall 35% at 16 engineers on related tasks, against 13% without sharing. On random tasks the drop is 7% vs 1%.
- **The no-sharing, related, n = 16 outlier is a runaway:** its 23K Muse tokens per engineer come from one answer that reasoned to Muse's 262,144-token limit (django__django-14580, which also ran away in `muse_direct`). Without it, it is ~6.9K per engineer, in line with the rest. Quote replay Muse tokens without runaways, or not at all.

**Claims the results support:**
- (a) On code, shared memory lets the closed model match full-context accuracy with < 10% of the context (prompt tokens).
- (b) On long conversations, raw retrieved pages recover the closed model's accuracy when retrieval finds the right session. Retrieval is the bottleneck.
- (c) Sharing memory across a team cuts per-engineer ingest work as the team grows.
- (d) The cheap model's confidence tells right drafts from wrong ones (AUC 0.65–0.80). A one-score cascade built on it beats random escalation by 5 points at 10% of the tokens on code.
- (e) On code, a team paying about a quarter of the tokens (about a third of the dollars) gets `muse_direct`'s accuracy (budget chosen after the run).
- (f) With a reasoning model, the closed model's own output, not the context, sets the cost of each escalation. That is what keeps a 10% total-token limit out of reach.